# Is Firm Facebook-Advertisable?

**Goal**: Tag companies founded/owned by entrepreneurs as likely **Facebook/Instagram advertisers** — i.e., B2C businesses whose product or service is *visual* or *experiential* and can be naturally promoted through a Facebook Page.

**What qualifies?**  
- Visual products: fashion, food, beauty, art, photography, home decor  
- Consumer experiences: restaurants, cafes, fitness studios, salons, events, travel  
- Consumer-facing digital brands: e-commerce, apps, media, subscription boxes  
- Local services people *choose* with emotion: wedding planners, photographers, gyms  

**What does NOT qualify?**  
- B2B/trade services (construction contractors, HVAC, trucking, accounting)  
- Legal / medical professional practices  
- Industrial / wholesale businesses  

**Approach**: Rule-based weighted keyword scoring on `company_name`, converted to a 0–1 confidence via sigmoid. Entirely local — no API calls.


## 1. Setup and Load Data

In [12]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Loading all_experience...', flush=True)
all_experience = pd.read_pickle('/shared/share_scp/coresignal/all_experience_AnalysisFile_latest.pkl')
print(f'Loaded {len(all_experience):,} rows, {all_experience.shape[1]} columns')
print('Columns:', list(all_experience.columns))

Loading all_experience...
Loaded 88,234,570 rows, 21 columns
Columns: ['member_id', 'title', 'company_name', 'company_url', 'date_from', 'date_to', 'duration', 'company_id', 'job_from', 'is_founder_only', 'is_owner_only', 'is_founder_or_owner', 'is_founder_or_owner_with_url', 'is_founder_or_owner_inc', 'is_cofounder_or_coowner_title', 'is_cofounder_only_title', 'is_coowner_only_title', 'long_title', 'franchise_founder_or_owner', 'franchise_owner', 'cofound_coown_same_url']


## 2. Filter to Entrepreneurs Only

In [13]:
# Determine which column flags entrepreneurs
if 'is_founder_or_owner' in all_experience.columns:
    entrepreneurs = all_experience[all_experience['is_founder_or_owner'] == True].copy()
elif 'is_founder_only' in all_experience.columns and 'is_owner_only' in all_experience.columns:
    entrepreneurs = all_experience[
        all_experience['is_founder_only'] | all_experience['is_owner_only']
    ].copy()
else:
    raise ValueError('No founder/owner flag found. Check column names above.')

print(f'Total entrepreneur rows     : {len(entrepreneurs):,}')
print(f'Unique company names        : {entrepreneurs["company_name"].nunique():,}')
print(f'Rows with null company_name : {entrepreneurs["company_name"].isna().sum():,}')

# Unique company names to score (deduplicated for speed)
unique_companies = (
    entrepreneurs['company_name']
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s != '']
    .unique()
)
print(f'Unique non-null company names to score: {len(unique_companies):,}')

Total entrepreneur rows     : 1,169,264
Unique company names        : 391,641
Rows with null company_name : 1,356
Unique non-null company names to score: 386,948


## 2 · Approach: BERT Embeddings + Supervised Learning

Instead of hand-crafted keyword rules, we use a data-driven approach:

1. **Sample 2,000 unique company names** at random and ask GPT-4o to label each as `yes` (likely Facebook/Instagram advertiser) or `no` (unlikely).
2. **Encode every name with BERT** (`all-MiniLM-L6-v2` via `sentence-transformers`), producing a 384-dimensional semantic embedding per name.
3. **Train a logistic regression** classifier on the 2,000-example labelled set, mapping embeddings → P(FB advertiser).
4. **Score all 386,948 unique company names** with the trained model and map probabilities to five ordered labels:

| P(yes) | Label |
|---|---|
| ≥ 0.80 | highly likely |
| 0.65 – 0.80 | likely |
| 0.35 – 0.65 | unclear |
| 0.20 – 0.35 | unlikely |
| &lt; 0.20 | highly unlikely |


In [27]:
# ── Install dependencies (only if missing) ───────────────────────────────────
import importlib, subprocess, sys

def _ensure(pkg, import_as=None):
    try:
        importlib.import_module(import_as or pkg)
    except ImportError:
        print(f"Installing {pkg}…")
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

_ensure('sentence-transformers', 'sentence_transformers')
_ensure('scikit-learn', 'sklearn')

# ── Core imports ──────────────────────────────────────────────────────────────
import re, json, time
import numpy as np
import pandas as pd
import openai

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

LABEL_ORDER = ['highly likely', 'likely', 'unclear', 'unlikely', 'highly unlikely']

print("All imports OK.")


Installing sentence-transformers…
All imports OK.


In [31]:
# ── Sample 1,000 unique company names ────────────────────────────────────────
rng = np.random.default_rng(42)
unique_companies = entrepreneurs['company_name'].dropna().unique()
sample_2000 = rng.choice(unique_companies, size=1000, replace=False).tolist()

# ── Label with GPT-4o (binary: yes / no) ─────────────────────────────────────
# Uses `client` already in kernel scope.
LABEL_SYSTEM = """\
You are a business classifier.  For each company name, decide whether this company
would be likely to run paid ads on Facebook or Instagram.

Answer "yes" for visual / experiential B2C businesses:
  retail shops, beauty salons, gyms/yoga studios, restaurants, bakeries, cafés,
  fashion & apparel brands, photographers, florists, event planners, home décor
  brands, travel agencies, pet groomers, children's activity centers, wellness
  coaches, art studios, entertainment venues, e-commerce shops, candle companies.

Answer "no" for B2B, industrial, or professional-services businesses:
  plumbers, HVAC contractors, freight/trucking, law firms, accounting firms,
  IT consulting, construction companies, manufacturers, wholesale distributors,
  staffing agencies, financial advisors, medical/dental practices.

When truly ambiguous, prefer "no".

Respond ONLY with a JSON object mapping each name exactly as given to "yes" or "no".
No extra keys, no commentary."""

gpt2000_labels: dict = {}
BATCH = 50
batches = [sample_2000[i:i+BATCH] for i in range(0, len(sample_2000), BATCH)]

print(f"Sending {len(sample_2000):,} names to GPT-4o in {len(batches)} batches…")

for idx, batch in enumerate(batches):
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model='gpt-4o',
                messages=[
                    {'role': 'system', 'content': LABEL_SYSTEM},
                    {'role': 'user',   'content': json.dumps(batch)},
                ],
                temperature=0,
                response_format={"type": "json_object"},
            )
            raw   = resp.choices[0].message.content.strip()
            chunk = json.loads(raw)
            gpt2000_labels.update(chunk)
            print(f"  Batch {idx+1}/{len(batches)}: {len(chunk)} labels")
            break
        except Exception as e:
            print(f"  Batch {idx+1} attempt {attempt+1} failed: {e}")
            time.sleep(3)

# Normalise keys and values
gpt2000_labels = {str(k).strip(): str(v).strip().lower()
                  for k, v in gpt2000_labels.items()}

yes_n = sum(1 for v in gpt2000_labels.values() if v == 'yes')
no_n  = sum(1 for v in gpt2000_labels.values() if v == 'no')
print(f"\nLabelled: {len(gpt2000_labels):,}  |  yes={yes_n:,}  no={no_n:,}")


Sending 1,000 names to GPT-4o in 20 batches…
  Batch 1/20: 50 labels
  Batch 2/20: 50 labels
  Batch 3/20: 50 labels
  Batch 4/20: 50 labels
  Batch 5/20: 50 labels
  Batch 6/20: 50 labels
  Batch 7/20: 50 labels
  Batch 8/20: 50 labels
  Batch 9/20: 50 labels
  Batch 10/20: 50 labels
  Batch 11/20: 50 labels
  Batch 12/20: 50 labels
  Batch 13/20: 50 labels
  Batch 14/20: 50 labels
  Batch 15/20: 50 labels
  Batch 16/20: 50 labels
  Batch 17/20: 50 labels
  Batch 18/20: 50 labels
  Batch 19/20: 50 labels
  Batch 20/20: 50 labels

Labelled: 1,000  |  yes=317  no=683


## 3 · Train BERT + Logistic Regression Classifier


In [32]:
# ── Load BERT model ───────────────────────────────────────────────────────────
print("Loading sentence-transformers model (all-MiniLM-L6-v2)…")
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

# ── Prepare training data (clean yes/no labels only) ─────────────────────────
train_names = [n for n in sample_2000 if gpt2000_labels.get(n) in ('yes', 'no')]
train_y     = np.array([1 if gpt2000_labels[n] == 'yes' else 0 for n in train_names])
print(f"Training examples: {len(train_names):,}  (yes={train_y.sum():,}  no={(1-train_y).sum():,})")

# ── Encode training names with BERT ──────────────────────────────────────────
print("Encoding training names with BERT…")
train_X = bert_model.encode(
    train_names, batch_size=256, show_progress_bar=True, normalize_embeddings=True
)

# ── Train logistic regression with 5-fold cross-validation ───────────────────
print("Training logistic regression (5-fold CV)…")
clf    = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
cv_auc = cross_val_score(clf, train_X, train_y, cv=5, scoring='roc_auc')
print(f"  CV AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")

clf.fit(train_X, train_y)
print("Model trained and ready.")


Loading sentence-transformers model (all-MiniLM-L6-v2)…


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 10066.96it/s]


Training examples: 986  (yes=310  no=676)
Encoding training names with BERT…


Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:03<00:00,  1.04it/s]

Training logistic regression (5-fold CV)…
  CV AUC: 0.882 ± 0.018
Model trained and ready.


In [35]:
# ── Hold-out test: draw 200 *new* names (never seen during training) ──────────
# Exclude the 1,000 training names from the pool
train_set      = set(sample_2000)
holdout_pool   = [n for n in unique_companies.tolist() if n not in train_set]
holdout_names  = rng.choice(holdout_pool, size=200, replace=False).tolist()

print(f"Sending 200 fresh names to GPT-4o for hold-out labels…")
holdout_labels: dict = {}
ho_batches = [holdout_names[i:i+50] for i in range(0, 200, 50)]
for idx, batch in enumerate(ho_batches):
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model='gpt-4o',
                messages=[
                    {'role': 'system', 'content': LABEL_SYSTEM},
                    {'role': 'user',   'content': json.dumps(batch)},
                ],
                temperature=0,
                response_format={"type": "json_object"},
            )
            chunk = json.loads(resp.choices[0].message.content.strip())
            holdout_labels.update(chunk)
            print(f"  Batch {idx+1}/4: {len(chunk)} labels")
            break
        except Exception as e:
            print(f"  Batch {idx+1} attempt {attempt+1} failed: {e}")
            time.sleep(3)

holdout_labels = {str(k).strip(): str(v).strip().lower()
                  for k, v in holdout_labels.items()}

# Encode and score the hold-out names with the trained model
clean_ho = [(n, holdout_labels[n]) for n in holdout_names
            if holdout_labels.get(n) in ('yes', 'no')]
ho_names, ho_true = zip(*clean_ho)
ho_y   = np.array([1 if v == 'yes' else 0 for v in ho_true])
ho_X   = bert_model.encode(list(ho_names), batch_size=256,
                            show_progress_bar=False, normalize_embeddings=True)
ho_probs = clf.predict_proba(ho_X)[:, 1]

from sklearn.metrics import roc_auc_score
ho_auc = roc_auc_score(ho_y, ho_probs)
print(f"\nHold-out test  (n={len(ho_y)}, yes={ho_y.sum()}, no={(1-ho_y).sum()})")
print(f"  AUC (out-of-sample): {ho_auc:.3f}")
print(f"  Train CV AUC       : {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")


Sending 200 fresh names to GPT-4o for hold-out labels…
  Batch 1/4: 50 labels
  Batch 2/4: 50 labels
  Batch 3/4: 50 labels
  Batch 4/4: 50 labels

Hold-out test  (n=199, yes=44, no=155)
  AUC (out-of-sample): 0.883
  Train CV AUC       : 0.882 ± 0.018


In [33]:
# ── Embed and score all unique company names ──────────────────────────────────
print(f"Encoding {len(unique_companies):,} unique company names with BERT…")
all_X = bert_model.encode(
    unique_companies.tolist(), batch_size=512,
    show_progress_bar=True, normalize_embeddings=True
)

# Predict P(yes = Facebook advertiser)
probs = clf.predict_proba(all_X)[:, 1]

def prob_to_label(p):
    if   p >= 0.80: return 'highly likely'
    elif p >= 0.65: return 'likely'
    elif p >= 0.35: return 'unclear'
    elif p >= 0.20: return 'unlikely'
    else:           return 'highly unlikely'

company_scores = pd.DataFrame({
    'company_name'     : unique_companies,
    'fb_ad_prob'       : probs,
    'fb_ad_likelihood' : pd.Categorical(
                             [prob_to_label(p) for p in probs],
                             categories=LABEL_ORDER, ordered=True
                         ),
})

print("\nDistribution of fb_ad_likelihood:")
for lbl in LABEL_ORDER:
    n = (company_scores['fb_ad_likelihood'] == lbl).sum()
    print(f"  {lbl:<20}: {n:8,}  ({n/len(company_scores):.1%})")


Encoding 391,641 unique company names with BERT…


Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 765/765 [29:14<00:00,  2.29s/it]



Distribution of fb_ad_likelihood:
  highly likely       :   14,983  (3.8%)
  likely              :   33,763  (8.6%)
  unclear             :  101,666  (26.0%)
  unlikely            :   85,510  (21.8%)
  highly unlikely     :  155,719  (39.8%)


In [34]:
# ── Merge scores onto entrepreneur rows ───────────────────────────────────────
_drop = [c for c in ['raw_score', 'confidence', 'fb_ad_likelihood',
                     'fb_ad_likelihood_final', 'matched_keywords', 'fb_ad_prob']
         if c in entrepreneurs.columns]
if _drop:
    entrepreneurs = entrepreneurs.drop(columns=_drop)

entrepreneurs = entrepreneurs.merge(
    company_scores[['company_name', 'fb_ad_prob', 'fb_ad_likelihood']],
    on='company_name',
    how='left'
)

print(f"Entrepreneurs with a score: {entrepreneurs['fb_ad_prob'].notna().sum():,}")
print("\nDistribution of fb_ad_likelihood across entrepreneur rows:")
for lbl in LABEL_ORDER:
    n = (entrepreneurs['fb_ad_likelihood'] == lbl).sum()
    print(f"  {lbl:<20}: {n:8,}  ({n/len(entrepreneurs):.1%})")


Entrepreneurs with a score: 1,167,908

Distribution of fb_ad_likelihood across entrepreneur rows:
  highly likely       :   40,864  (3.5%)
  likely              :   93,036  (8.0%)
  unclear             :  298,599  (25.5%)
  unlikely            :  263,075  (22.5%)
  highly unlikely     :  472,334  (40.4%)


## 4 · Random Sample for Manual Validation

Review 40 rows from each label category (200 total).  
Ask: *"Is `fb_ad_likelihood` the right label for this company?"*

| Column | Meaning |
|---|---|
| `company_name` | Raw name from LinkedIn |
| `fb_ad_likelihood` | Predicted label (5-level) |
| `fb_ad_prob` | P(Facebook advertiser) from the BERT + logistic regression model |


In [ ]:
import random
random.seed(42)
np.random.seed(42)

# Draw up to 40 names from each of the 5 categories → up to 200 rows total
N_PER_LABEL = 40
frames = []
for label in LABEL_ORDER:
    subset = company_scores[company_scores['fb_ad_likelihood'] == label]
    n = min(N_PER_LABEL, len(subset))
    if n > 0:
        frames.append(subset.sample(n=n, random_state=42))
        print(f"  {label:<20}: sampled {n} of {len(subset):,}")

validation_sample = (
    pd.concat(frames)
    .sample(frac=1, random_state=99)   # shuffle so labels are interleaved
    .reset_index(drop=True)
)

display_cols = ['company_name', 'fb_ad_likelihood', 'fb_ad_prob']

print(f"\nTotal validation sample: {len(validation_sample)} rows")
print("Review the table and note mis-classifications.\n")

pd.set_option('display.max_rows', 210)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.3f}'.format)

validation_sample[display_cols]


  highly likely      : sampled 40 of 25,375
  likely             : sampled 40 of 7,213
  unclear            : sampled 40 of 348,072
  unlikely           : sampled 40 of 1,902
  highly unlikely    : sampled 40 of 4,386

Total validation sample: 200 rows
Review the table and note mis-classifications.



,company_name,fb_ad_likelihood,confidence,raw_score,matched_keywords
0,Sigmon Wealth Management,unlikely,0.269,-1.000,'Wealth Management'(-1.0)
1,"Heritage Asset Management, LLC",unlikely,0.269,-1.000,'Asset Management'(-1.0)
2,Pats Cleaning Service,highly unlikely,0.119,-2.000,'Cleaning Service'(-2.0)
3,Knewsco Digital Inc.,likely,0.690,0.800,'Digital'(+0.8)
4,Tidus Consulting,unclear,0.500,0.000,(none)
5,L&C Landholdings LLC,unclear,0.500,0.000,(none)
6,Rylie Doo’s,unclear,0.500,0.000,(none)
7,"Barnett Marketing and Sales, LLC",highly likely,0.818,1.500,'Marketing'(+1.5)
8,IT Consulting Services,unlikely,0.269,-1.000,'IT Consulting'(-1.0)
9,Coastal Legal Counsel,highly unlikely,0.182,-1.500,'Counsel'(-1.5)


## 5 · GPT-4o Validation — Independent Classification of the 200-Name Sample

Send the validation sample to GPT-4o and ask it to assign the same 5-level label independently.  
Compare GPT's answers against the BERT model's labels, print every disagreement, and report the agreement rate.

**Workflow:**
1. Run the **API call cell** — sends all 200 names to GPT-4o in batches of 50
2. Run the **comparison cell** — prints agreement rate and lists every disagreement


In [18]:
import openai
import json
import time

# ── API key ───────────────────────────────────────────────────────────────────
# Rotate this key at https://platform.openai.com/api-keys after use.
# import os  # (already imported above)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
client = openai.OpenAI(api_key=OPENAI_API_KEY)

VALID_LABELS = ['highly likely', 'likely', 'unclear', 'unlikely', 'highly unlikely']

SYSTEM_PROMPT = """You are a business analyst helping classify company names.

The task: given a list of company names, decide how likely each company is to run
Facebook / Instagram paid advertising — i.e. does it sell a *visual* or *experiential*
consumer product or service that would make a natural Facebook Page?

Examples that are POSITIVE (run FB ads):
  fashion boutiques, beauty salons, restaurants, bakeries, gyms/yoga studios,
  photographers, florists, event planners, home decor brands, travel agencies,
  e-commerce shops, candle companies, pet groomers, children's activity centers.

Examples that are NEGATIVE (unlikely to run FB ads):
  plumbers, HVAC contractors, freight/trucking companies, law firms, accounting
  firms, B2B software companies, medical practices, industrial manufacturers,
  janitorial services, funeral homes.

Assign each company EXACTLY one of these five labels (use the exact string):
  highly likely | likely | unclear | unlikely | highly unlikely

Return ONLY a JSON array of objects with keys "company_name" and "label".
No explanation, no markdown fences, just the raw JSON array."""

def classify_batch(names, batch_size=50, pause=1.0):
    """Send names to GPT-4o in batches; return {name: label} dict."""
    results = {}
    batches = [names[i:i+batch_size] for i in range(0, len(names), batch_size)]
    for batch_idx, batch in enumerate(batches):
        numbered = '\n'.join(f'{i+1}. {n}' for i, n in enumerate(batch))
        user_msg = (
            f'Classify these {len(batch)} company names:\n\n{numbered}\n\n'
            'Return a JSON array: [{"company_name": "...", "label": "..."}, ...]'
        )
        for attempt in range(3):
            try:
                resp = client.chat.completions.create(
                    model='gpt-4o',
                    messages=[
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user',   'content': user_msg},
                    ],
                    temperature=0,
                    max_tokens=4096,
                )
                raw = resp.choices[0].message.content.strip()
                # Strip accidental markdown fences
                if raw.startswith('```'):
                    raw = '\n'.join(raw.split('\n')[1:])
                if raw.endswith('```'):
                    raw = '\n'.join(raw.split('\n')[:-1])
                parsed = json.loads(raw)
                for item in parsed:
                    name  = item.get('company_name', '').strip()
                    label = item.get('label', '').strip().lower()
                    if label not in VALID_LABELS:
                        label = 'unclear'
                    results[name] = label
                print(f'  Batch {batch_idx+1}/{len(batches)}: {len(parsed)} labels received')
                break
            except Exception as e:
                print(f'  Batch {batch_idx+1} attempt {attempt+1} failed: {e}')
                time.sleep(3)
        time.sleep(pause)
    return results

# ── Run classification ────────────────────────────────────────────────────────
names_to_classify = validation_sample['company_name'].tolist()
print(f'Sending {len(names_to_classify)} names to GPT-4o...')
gpt_labels = classify_batch(names_to_classify)

# Attach GPT labels to validation sample
validation_sample['gpt_label'] = validation_sample['company_name'].map(gpt_labels)
print(f'\nGPT labels received for {validation_sample["gpt_label"].notna().sum()} / {len(validation_sample)} names')
print('\nGPT label distribution:')
print(validation_sample['gpt_label'].value_counts().reindex(VALID_LABELS).to_string())


Sending 200 names to GPT-4o...
  Batch 1/4: 50 labels received
  Batch 2/4: 50 labels received
  Batch 3/4: 50 labels received
  Batch 4/4: 50 labels received

GPT labels received for 200 / 200 names

GPT label distribution:
gpt_label
highly likely      24
likely             22
unclear            45
unlikely           42
highly unlikely    67


In [19]:
# ── Compare BERT model vs GPT-4o ─────────────────────────────────────────────
compare = validation_sample[['company_name', 'fb_ad_likelihood', 'gpt_label', 'fb_ad_prob']].copy()
compare = compare[compare['gpt_label'].notna()]
compare['agree'] = compare['fb_ad_likelihood'] == compare['gpt_label']

n_total = len(compare)
n_agree = compare['agree'].sum()
n_wrong = n_total - n_agree
pct_ok  = n_agree / n_total * 100

print('=' * 60)
print(f'  Total compared  : {n_total}')
print(f'  Model agree     : {n_agree}  ({pct_ok:.1f}%)')
print(f'  Disagreements   : {n_wrong}  ({100 - pct_ok:.1f}%)')
print('=' * 60)

# ── Disagreement detail ───────────────────────────────────────────────────────
wrong = compare[~compare['agree']].copy()

if len(wrong) == 0:
    print('\nPerfect agreement — no disagreements!')
else:
    print(f'\n{"#":<4}  {"Company Name":<40}  {"BERT Model":<18}  {"GPT-4o":<18}  P(yes)')
    print('-' * 95)
    for i, (_, row) in enumerate(wrong.iterrows(), 1):
        print(f'{i:<4}  {str(row["company_name"]):<40}  '
              f'{row["fb_ad_likelihood"]:<18}  {row["gpt_label"]:<18}  {row["fb_ad_prob"]:.3f}')

    print()
    print('Disagreement breakdown by direction:')
    direction = wrong.groupby(['fb_ad_likelihood', 'gpt_label']).size().rename('count').reset_index()
    direction.columns = ['model_said', 'gpt_said', 'count']
    direction = direction.sort_values('count', ascending=False)
    print(direction.to_string(index=False))


  Total compared  : 200
  Algorithm agree : 57  (28.5%)
  Disagreements   : 143  (71.5%)

#     Company Name                              Algorithm           GPT-4o              Matched keywords
----------------------------------------------------------------------------------------------------------------------------------
1     Sigmon Wealth Management                  unlikely            highly unlikely     'Wealth Management'(-1.0)
2     Heritage Asset Management, LLC            unlikely            highly unlikely     'Asset Management'(-1.0)
3     Pats Cleaning Service                     highly unlikely     unlikely            'Cleaning Service'(-2.0)
4     Knewsco Digital Inc.                      likely              unclear             'Digital'(+0.8)
5     Tidus Consulting                          unclear             highly unlikely     (none)
6     L&C Landholdings LLC                      unclear             highly unlikely     (none)
7     Rylie Doo’s                       

## 6 · Manual Corrections

After reviewing the validation sample, add mis-classified names to `FORCE_LABEL` below.  
Valid labels: `'highly likely'`, `'likely'`, `'unclear'`, `'unlikely'`, `'highly unlikely'`.  
Re-run this cell and Section 4 onward to see updated results.


In [ ]:
# ── Manual overrides ─────────────────────────────────────────────────────────
# Map company_name → correct label (one of LABEL_ORDER)
# Example:
#   'Acme Photo Studio'  : 'highly likely',
#   'Smith Plumbing LLC' : 'highly unlikely',

FORCE_LABEL = {
    # 'Company Name Here': 'highly likely',
}

# ── Apply overrides ───────────────────────────────────────────────────────────
company_scores['fb_ad_likelihood_final'] = company_scores['fb_ad_likelihood'].copy()

if FORCE_LABEL:
    for name, label in FORCE_LABEL.items():
        if label not in LABEL_ORDER:
            print(f'WARNING: invalid label "{label}" for "{name}" — skipping')
            continue
        mask = company_scores['company_name'] == name
        if mask.sum() == 0:
            print(f'WARNING: "{name}" not found in company_scores — skipping')
        else:
            company_scores.loc[mask, 'fb_ad_likelihood_final'] = label

changed = (company_scores['fb_ad_likelihood_final'] != company_scores['fb_ad_likelihood']).sum()
print(f'Overrides applied: {changed}')
print()
print('Updated distribution:')
counts = company_scores['fb_ad_likelihood_final'].value_counts().reindex(LABEL_ORDER)
for label, n in counts.items():
    pct = n / len(company_scores) * 100
    print(f'  {label:<20}: {n:8,}  ({pct:5.1f}%)')


Overrides applied: 0

Updated distribution:
  highly likely      :   25,375  (  6.6%)
  likely             :    7,213  (  1.9%)
  unclear            :  348,072  ( 90.0%)
  unlikely           :    1,902  (  0.5%)
  highly unlikely    :    4,386  (  1.1%)


## 7 · Sanity Check — Top and Bottom Companies by P(yes)


In [25]:
show_cols = ['company_name', 'fb_ad_likelihood', 'fb_ad_prob']

print('=== TOP 30 most confidently HIGHLY LIKELY companies ===')
display(
    company_scores[company_scores['fb_ad_likelihood'] == 'highly likely']
    .sort_values('fb_ad_prob', ascending=False)
    .head(30)[show_cols]
    .reset_index(drop=True)
)

print('\n=== TOP 30 most confidently HIGHLY UNLIKELY companies ===')
display(
    company_scores[company_scores['fb_ad_likelihood'] == 'highly unlikely']
    .sort_values('fb_ad_prob', ascending=True)
    .head(30)[show_cols]
    .reset_index(drop=True)
)

print('\n=== Sample of UNCLEAR companies (closest to 0.50 probability) ===')
display(
    company_scores[company_scores['fb_ad_likelihood'] == 'unclear']
    .assign(_dist=lambda d: (d['fb_ad_prob'] - 0.5).abs())
    .sort_values('_dist')
    .head(30)[show_cols]
    .reset_index(drop=True)
)


=== TOP 30 most confidently HIGHLY LIKELY companies ===


,company_name,fb_ad_likelihood,confidence,raw_score,matched_keywords
0,"Atlanta SEO Pro, LLC - An Atlanta, GA SEO Company and Full Service Digital M...",highly likely,1.000,12.800,'Digital Marketing'(+3.5); 'SEO'(+3.5); 'Marketing Agency'(+3.5); 'Marketing...
1,Invent | SEO | Social Media Marketing | Content Driven Solutions | Digital M...,highly likely,1.000,11.300,'Social Media Marketing'(+3.5); 'SEO'(+3.5); 'Social Media'(+2.0); 'Marketin...
2,Oakwood Lane (eCommerce & Social Media Marketing Consulting),highly likely,1.000,10.000,'Social Media Marketing'(+3.5); 'eCommerce'(+3.0); 'Social Media'(+2.0); 'Ma...
3,Vertical Web - Digital Marketing Agency,highly likely,1.000,9.300,'Digital Marketing'(+3.5); 'Marketing Agency'(+3.5); 'Marketing'(+1.5); 'Web...
4,"Jolly Sherpa, Digital Marketing Agency",highly likely,1.000,9.300,'Digital Marketing'(+3.5); 'Marketing Agency'(+3.5); 'Marketing'(+1.5); 'Dig...
5,SEO & Digital Marketing,highly likely,1.000,9.300,'Digital Marketing'(+3.5); 'SEO'(+3.5); 'Marketing'(+1.5); 'Digital'(+0.8)
6,New England Digital Marketing Agency,highly likely,1.000,9.300,'Digital Marketing'(+3.5); 'Marketing Agency'(+3.5); 'Marketing'(+1.5); 'Dig...
7,Permian Edge Digital Marketing Agency,highly likely,1.000,9.300,'Digital Marketing'(+3.5); 'Marketing Agency'(+3.5); 'Marketing'(+1.5); 'Dig...
8,Digital Marketing Agency,highly likely,1.000,9.300,'Digital Marketing'(+3.5); 'Marketing Agency'(+3.5); 'Marketing'(+1.5); 'Dig...
9,"MSE, LLC. | Digital Marketing Agency",highly likely,1.000,9.300,'Digital Marketing'(+3.5); 'Marketing Agency'(+3.5); 'Marketing'(+1.5); 'Dig...



=== TOP 30 most confidently HIGHLY UNLIKELY companies ===


,company_name,fb_ad_likelihood,confidence,raw_score,matched_keywords
0,"Lewis Lawn Care and Masonry, LLC",highly unlikely,0.007,-5.000,'Lawn Care'(-2.5); 'Masonry'(-2.5)
1,"LATHAM LAWN CARE AND PRESSURE WASH, LLC",highly unlikely,0.011,-4.500,'LAWN CARE'(-2.5); 'PRESSURE WASH'(-2.0)
2,Showcase Lawn Care & Services DBA Frankie B's Hauling,highly unlikely,0.011,-4.500,'Lawn Care'(-2.5); 'Hauling'(-2.0)
3,"O & W Mortuary Services, PLLC",highly unlikely,0.047,-3.000,'Mortuary'(-3.0)
4,Sterling Lewis Funeral Home,highly unlikely,0.047,-3.000,'Funeral'(-3.0)
5,"Ponte Vedra Valley: Funeral Home, Cremation Center & Cemetery",highly unlikely,0.047,-3.000,'Funeral'(-3.0)
6,THE FAT FUNERAL ®™,highly unlikely,0.047,-3.000,'FUNERAL'(-3.0)
7,Potti & Marc F. Burr Funeral Homes,highly unlikely,0.047,-3.000,'Funeral'(-3.0)
8,Christians Funeral Home,highly unlikely,0.047,-3.000,'Funeral'(-3.0)
9,Daniels Chapel of the Roses Funeral Home & Crematory,highly unlikely,0.047,-3.000,'Funeral'(-3.0)



=== Sample of UNCLEAR companies (closest to 0.50 confidence) ===


,company_name,fb_ad_likelihood,confidence,raw_score,matched_keywords
0,Karla Consulting Services,unclear,0.500,0.000,(none)
1,Wheelhouse Press,unclear,0.500,0.000,(none)
2,PlusAI,unclear,0.500,0.000,(none)
3,Locus Energy,unclear,0.500,0.000,(none)
4,Meditation Without Borders,unclear,0.500,0.000,(none)
5,"Black Border Media, LLC",unclear,0.500,0.000,(none)
6,"M.A.W. Consulting, LLC",unclear,0.500,0.000,(none)
7,ElegrootSolutions LLC,unclear,0.500,0.000,(none)
8,Versatile Image,unclear,0.500,0.000,(none)
9,Kenneth Kendall Coaching,unclear,0.500,0.000,(none)


## 8. Save Results

In [26]:
# Merge final label back onto entrepreneurs
_drop = [c for c in ['fb_ad_likelihood', 'fb_ad_prob'] if c in entrepreneurs.columns]
if _drop:
    entrepreneurs = entrepreneurs.drop(columns=_drop)

entrepreneurs = entrepreneurs.merge(
    company_scores[['company_name', 'fb_ad_prob', 'fb_ad_likelihood_final']]
    .rename(columns={'fb_ad_likelihood_final': 'fb_ad_likelihood'}),
    on='company_name',
    how='left'
)

print('Final distribution across entrepreneur rows:')
counts = entrepreneurs['fb_ad_likelihood'].value_counts().reindex(LABEL_ORDER)
for label, n in counts.items():
    pct = n / len(entrepreneurs) * 100
    print(f'  {label:<20}: {n:8,}  ({pct:5.1f}%)')

out_path = '/shared/share_scp/coresignal/entrepreneurs_marketing_oriented_latest.pkl'
entrepreneurs.to_pickle(out_path)
print(f'\nSaved {len(entrepreneurs):,} rows to {out_path}')

scores_path = '/shared/share_scp/coresignal/company_marketing_scores_latest.pkl'
company_scores.to_pickle(scores_path)
print(f'Saved company scores ({len(company_scores):,} unique names) to {scores_path}')


Final distribution across entrepreneur rows:
  highly likely      :   70,884  (  6.1%)
  likely             :   25,550  (  2.2%)
  unclear            : 1,047,496  ( 89.6%)
  unlikely           :    5,412  (  0.5%)
  highly unlikely    :   10,702  (  0.9%)

Saved 1,169,264 rows to /shared/share_scp/coresignal/entrepreneurs_marketing_oriented_latest.pkl
Saved company scores (386,948 unique names) to /shared/share_scp/coresignal/company_marketing_scores_latest.pkl
